In [ ]:
# ══════════════════════════════════════════════
# CELDA 1 — Conectar Google Drive
# ══════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════
# CELDA 2 — Instalar dependencias
# ══════════════════════════════════════════════
!pip install ultralytics roboflow -q
print("✅ Dependencias instaladas")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 86.0 MB/s eta 0:00:00
✅ Dependencias instaladas


In [ ]:
# ══════════════════════════════════════════════
# CELDA 3 — Descargar dataset
# ══════════════════════════════════════════════
from roboflow import Roboflow

rf = Roboflow(api_key="6Ts0niJMvk2Dlr8Wcngv")
rf.workspace("activity-graz-uni") \
  .project("volleyball-activity-dataset") \
  .version(3) \
  .download("yolov8", location="/content/data/events")

print("✅ Dataset descargado")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data/events in yolov8:: 100%|██████████| 50012/50012 [00:49<00:00, 1015.65it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset descargado


In [ ]:
# ══════════════════════════════════════════════
# CELDA 4 — Verificar que el modelo existe en Drive
# ══════════════════════════════════════════════
import os

modelo = "/content/drive/MyDrive/VolleyVision/volleyvision_events_best_final.pt"

if os.path.exists(modelo):
    size = os.path.getsize(modelo) / 1024 / 1024
    print(f"✅ Modelo encontrado: {size:.1f} MB")
else:
    print("❌ No se encuentra el modelo. Verifica la ruta en Drive.")

✅ Modelo encontrado: 5.9 MB


In [ ]:
# ══════════════════════════════════════════════
# CELDA 5 — Fine-tuning con augmentations
# ══════════════════════════════════════════════
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/VolleyVision/volleyvision_events_best_final.pt")

print("🚀 Iniciando entrenamiento...")

results = model.train(
    data    = "/content/data/events/data.yaml",
    epochs  = 30,
    imgsz   = 640,
    batch   = 16,
    workers = 4,
    device  = 0,
    patience = 15,
    project  = "/content/runs/detect/models",
    name     = "volleyvision_multiangle",
    exist_ok = True,

    # ── augmentations para múltiples ángulos ──
    perspective  = 0.0005,
    degrees      = 12.0,
    shear        = 8.0,
    scale        = 0.6,
    translate    = 0.15,
    fliplr       = 0.5,
    flipud       = 0.05,
    hsv_h        = 0.015,
    hsv_s        = 0.5,
    hsv_v        = 0.4,
    mosaic       = 1.0,
    mixup        = 0.15,
    copy_paste   = 0.1,
    erasing      = 0.3,
)

print("✅ Entrenamiento completado")

🚀 Iniciando entrenamiento...
Ultralytics 8.4.79 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/events/data.yaml, degrees=12.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.3, exist_ok=True, fliplr=0.5, flipud=0.05, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/content/drive/MyDrive/VolleyVision/volleyvision_events_best_final.pt, momentum=0.937, mosaic=1.0, multi_s

In [ ]:
# ══════════════════════════════════════════════
# CELDA 6 — Guardar modelos en Drive
# ══════════════════════════════════════════════
import shutil

base = "/content/runs/detect/models/volleyvision_multiangle/weights"
dest = "/content/drive/MyDrive/VolleyVision"

shutil.copy(f"{base}/best.pt", f"{dest}/volleyvision_multiangle_best.pt")
shutil.copy(f"{base}/last.pt", f"{dest}/volleyvision_multiangle_last.pt")

print("✅ Guardados en Drive:")
print("   → volleyvision_multiangle_best.pt")
print("   → volleyvision_multiangle_last.pt")

In [ ]:
# ══════════════════════════════════════════════
# CELDA 7 — Evaluar en test set
# ══════════════════════════════════════════════
from ultralytics import YOLO

model_final = YOLO("/content/drive/MyDrive/VolleyVision/volleyvision_multiangle_best.pt")

metrics = model_final.val(
    data  = "/content/data/events/data.yaml",
    split = "test",
    conf  = 0.4,
    iou   = 0.5
)

print(f"\n📊 Resultados vs modelo anterior:")
print(f"{'Métrica':<12} {'Nuevo':>8} {'Anterior':>10}")
print(f"{'─'*32}")
print(f"{'mAP50':<12} {metrics.box.map50:>8.3f} {'0.972':>10}")
print(f"{'mAP50-95':<12} {metrics.box.map:>8.3f} {'0.889':>10}")
print(f"{'Precision':<12} {metrics.box.mp:>8.3f} {'0.969':>10}")
print(f"{'Recall':<12} {metrics.box.mr:>8.3f} {'0.977':>10}")